In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, TimestampType, MapType

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "", "catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")

catalog_name = dbutils.widgets.get("catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")

In [0]:
# Create schema for dq_rule_definitions
dqx_rule_definitions_schema = StructType([
    StructField("rule_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("rule_function", StringType(), False),  # is_not_null, is_unique, etc.
    StructField("description", StringType(), True),
    StructField("rule_type", StringType(), False),  # column_level, row_level, aggregate
    StructField("rule_dimension", StringType(), False), 
    StructField("created_date", TimestampType(), False),
    StructField("updated_date", TimestampType(), False),
    StructField("argument_placeholder", StringType(), False),
    StructField("is_arg_mendatory", BooleanType(), False)
])
# Create Delta table and overwrite schema (mergeSchema enabled)
spark.createDataFrame([], dqx_rule_definitions_schema) \
    .write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{config_schema_name}.dqx_rule_definitions")

print("✓ Created table: dqx_rule_definitions")

In [0]:
# Create schema for dq_rule_configs (main configuration table)
dqx_rule_mappings_schema = StructType([
    StructField("table_name", StringType(), False),          # Accept duplicate but not null
    StructField("rule_id", StringType(), False),         # e.g., 'customer_id_not_null'
    StructField("column_name", StringType(), True),        # e.g., 'customer_id'
    StructField("criticality", StringType(), False),       # 'error' or 'warn'
    StructField("is_active", BooleanType(), False),
    StructField("arguments", MapType(StringType(), StringType()), True),  # for complex rules like range: {'limit': '0'}
    StructField("updated_at", TimestampType(), False)
])

# Create Delta table partitioned by table_name
spark.createDataFrame([], dqx_rule_mappings_schema) \
    .write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .partitionBy("table_name") \
    .saveAsTable(f"{catalog_name}.{config_schema_name}.dqx_rule_mappings")

print("✓ Created table: dqx_rule_mappings (partitioned by table_name)")